## Imports and Functions

In [ ]:
!uv pip install matplotlib

In [ ]:
from pathlib import Path
import polars as pl
from polars import selectors as cs
import numpy as np
import altair as alt
import xgboost as xgb
import json
from pathlib import Path
import wandb
from wandb.integration.xgboost import WandbCallback
import wandb_workspaces.reports.v2 as wr
from criteo_functions import *

#import cupy as cp
pl.Config.set_tbl_rows(100)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
data_dir = Path("~/projects/data/Criteo").expanduser()
data_dir = Path("/workspace/data/Criteo")
data_types = ("train", "valid", "test")
paths = [data_dir / f"{data_type}.parquet" for data_type in data_types ]

enums = build_categorical_enums(paths)


n_ints = 13
ints = [f"I{i}" for i in range(1,n_ints+1)]
n_cats = 26
cats = [f"C{i}" for i in range(1,n_cats+1)]
    


## Analysis

> **⚠️ Broken cell(s) below** — `c1`, `rest`, `gm` are not defined by any earlier cell in this notebook; the cell that originally created them was apparently deleted. To fix, add a cell above with something like:
>
> ```python
> config = {'cat_cutoff': 25, 'cat_pseudo_count': 16}
> c1, rest, gm = transform_cat(train, 'C1', 'Label', cutoff=config['cat_cutoff'], pseudo_count=config['cat_pseudo_count'])
> ```
>
> (requires `train` from the 'Load data' section below to already be loaded). This affects this cell and the two CTR-plot cells further down that also reference `c1`/`rest`/`gm`.

In [ ]:
print(gm,rest.collect())
c1.head().collect()

#### Integer percentiles

In [ ]:
train.select(ints).describe(percentiles=np.linspace(0,1,21))

#### Categorical Nulls

In [ ]:
train.select(pl.col(cats).is_null()).sum().collect()

In [ ]:
train.head().collect()

#### Categorical CTR plot

In [ ]:

base_chart = alt.Chart(c1.head(50).collect())
ctr_points = (
    base_chart    
    .mark_point()
    .transform_fold(['ctr', 'ctr_smooth'], as_=['smooth','ctr'])
    .encode(x='C1', y='ctr',color='smooth:N')
)
rest_rule = base_chart.mark_rule(strokeDash=[2,2]).encode(y=alt.datum(rest.collect()['ctr_smooth'][0]))
gm_rule = base_chart.mark_rule(strokeDash=[2,2]).encode(y=alt.datum(gm))
ctr_points + rest_rule + gm_rule


> **⚠️** `df` is not defined at this point in the notebook (it's only assigned later, in the Tree Analysis section, to something unrelated). Define it first, e.g. `df = train.collect()`, if you want to run this cell.

In [ ]:
df.select(cs.starts_with('C12')).head().collect()

In [ ]:
train.select('Label').describe()

> **⚠️** `cat_encode` is not defined at this point. It's built by `calc_cat_encode(train, config, cats)` inside `get_data()`, but not returned/exposed there. To inspect it here, call `cat_encode, gm = calc_cat_encode(train, config, cats)` directly first.

In [ ]:
cat_encode['C20']

In [ ]:
df_plot = c1.head(50).collect()
alt.Chart(df_plot).transform_fold(['diff','margin'], as_=['plot','ctr']).encode(x='C1',y='ctr',color='plot:N').mark_point()

### Frequent Items

In [ ]:
sample_every = 10
train_sample = pl.scan_parquet("~/projects/data/Criteo/train.parquet", row_index_name='Row').filter(pl.col('Row').mod(sample_every)==0)


In [ ]:
train_sample.select([pl.col(f"I{d}").qcut(20, allow_duplicates=True).cast(pl.String).fill_null("NA") for d in range(1,14)]).collect() 

In [ ]:
train_sample.head().collect()

In [ ]:
items = (
    train_sample
    .select(["Label",*cats])
    .fill_null("missing")
    .select([pl.concat_str(pl.lit(cat),pl.col(cat),separator=":").alias(cat)  for cat in ["Label",*cats]])
)

In [ ]:
items.head().collect()

In [ ]:
items.sink_csv("train_sample_100_items.tsv", separator="\t")

In [ ]:
!head train_sample_1000_items.tsv

`eclat -tr -s10 -eclg -R appearances.txt train_sample_1000_items.tsv train_sample_1000_items.txt`

In [ ]:
!eclat -tr  -eclg -c25 -R appearances.txt train_sample_10_items.tsv train_sample_10_associations.txt

In [ ]:
!eclat -ts -s-1000 train_sample_10_items.tsv train_sample_10_items.txt

```
C9:a73ee510 (89.7093)
C22:missing C9:a73ee510 (68.6181)
C22:missing (76.4138)
Label:0 C9:a73ee510 (65.3188)
Label:0 C22:missing C9:a73ee510 (50.8235)
Label:0 C22:missing (57.6457)
Label:0 (74.2651)
```

In [ ]:
ar.shape

In [ ]:
ff = parse_frequent("train_sample_1000_items.txt").sort("support",descending=True).head(30).collect()

In [ ]:
ff["rules"][1]

In [ ]:
ar.shape

In [ ]:
ar_rules = {f"ar_{i}": ar_rule for i, ar_rule in enumerate(ar.get_column("rules").to_list())}

In [ ]:
train_ = (
    pl.scan_parquet("/workspace/data/Criteo/train.parquet")
    .with_columns(cs.starts_with("C").fill_null("missing"))
    .with_columns(**ar_rules)
    .sink_parquet("/workspace/data/Criteo/train_assoc997_copy.parquet")
)

In [ ]:
train_sample.group_by("C9").agg(pl.len()).collect()

## Load data

In [ ]:
%%time
config = {'sample_every': 1, 'cat_cutoff': 25, 'cat_pseudo_count':16}
train_X, train_y, valid_X, valid_y, test_X, test_y = get_data(config)


In [ ]:
train, valid, test = load_data(1)

In [ ]:
v1 = valid.collect()

In [ ]:
v1.estimated_size(unit="gigabytes")

In [ ]:
t1 = train.collect()

In [ ]:
t1.dtypes

In [ ]:
t1.estimated_size('gb')

using enums instead of strings does not cause any memory efficiency (according to estimated_size)
estimate train at 10 gb and train with enums at 6gb ( beased on sampling every 10th row)


In [ ]:
t2 = train.with_columns([pl.col(c).cast(dt) for c, dt in enums.items()]).collect()

In [ ]:
t2.estimated_size('gb')

In [ ]:
v2 = v1.with_columns([pl.col(c).cast(dt) for c, dt in enums.items()])

In [ ]:
v2.estimated_size('gb')

In [ ]:
v2.head()

In [ ]:
ctr_C9 = create_ctr(train, "C9")

In [ ]:
ctr_C9.head(10).collect()

## Load Association Rules

In [ ]:
ar = parse_association("train_sample_10_associations.txt").collect()

In [ ]:
ar_rules = {f"ar_{i}": ar_rule for i, ar_rule in enumerate(ar.get_column("rules").to_list())}

In [ ]:
ar.sort("confidence", descending=True)

25% CTR 36 million rows originally

In [ ]:
train_X.head()

In [ ]:
train_X, train_y, valid_X, valid_y = get_data(config)

In [ ]:
config["association_rules"]=150

In [ ]:

train_X = (
    train_X
    .drop(
        [f"ar_{i}" for i in range(config["association_rules"],997)]
    ,strict=False)
)
valid_X = (
    valid_X
    .drop(
        [f"ar_{i}" for i in range(config["association_rules"],997)]
    ,strict=False)
)


In [ ]:
train_X.shape

In [ ]:
train_X.head()

In [ ]:
train_X.shape, valid_X.shape

## Train Model

### Train with Association Rules

In [ ]:
for n_rules in [30,20,10,5,4,3,2]:
    config["association_rules"]=n_rules
    print(n_rules)
    train_X = (
        train_X
        .drop(
            [f"ar_{i}" for i in range(config["association_rules"],997)]
        ,strict=False)
    )
    valid_X = (
        valid_X
        .drop(
            [f"ar_{i}" for i in range(config["association_rules"],997)]
        ,strict=False)
    )
    
    with wandb.init(project="criteo_boost_sample_ar_10",config=config) as run:
        clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=10, reg_lambda=0, gamma=10, n_estimators=200,learning_rate=.1,max_depth=60, callbacks=[WandbCallback(log_feature_importance=True)])
        clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    #tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
with wandb.init(project="criteo_boost_sample_ar_10",config=config) as run:
    clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=10, reg_lambda=0, gamma=10, n_estimators=200,learning_rate=.1,max_depth=60, callbacks=[WandbCallback(log_feature_importance=True)])
    clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    #tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
with wandb.init(project="criteo_boost_sample_ar_10",config=config) as run:
    clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=10, reg_lambda=0, gamma=10, n_estimators=200,learning_rate=.1,max_depth=60, callbacks=[WandbCallback(log_feature_importance=True)])
    clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    #tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=0, n_estimators=20)
clf.fit(train_X, train_y, eval_set=[(valid_X, valid_y)],verbose=2,)
tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=10, n_estimators=150,learning_rate=.1,max_depth=20, callbacks=[WandbCallback(log_feature_importance=True)])
clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
tree_df = clf.get_booster().trees_to_dataframe()

## Tree Analysis

In [ ]:
clf.intercept_

In [ ]:
tree_df1 = pl.from_pandas(tree_df)

In [ ]:
clf.get_xgb_params()

In [ ]:
xgb.plot_importance(clf)

In [ ]:
xgb.importance(clf)

In [ ]:
tree_df1.get_column('Node').describe(percentiles=[.1,.2,.3,.4,.5,.6,.7,.8,.9,.95,.99])

In [ ]:
tree_df1.select('Cover').describe(percentiles=[.1,.2,.3,.4,.5,.6,.7,.8,.9])

In [ ]:

clf.get_booster().save_config()

In [ ]:
tree_df1.filter(pl.col('Tree')==0)

Cover is the sum of Hessians

Gain is the split improvement score

$$Gain=\frac{1}{2}\left[\frac{G^2_L}{H_L+\lambda} +  \frac{G^2_R}{H_R+\lambda} - \frac{(G_L + G_R)^2}{H_L+H_R + \lambda}\right]-\gamma $$

### Breast Cancer Test Data

In [ ]:
95528.6328 + 254078.5620

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

import xgboost as xgb

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=94)

# Use "hist" for constructing the trees, with early stopping enabled.
clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2)
# Fit the model, test sets are used for early stopping.
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)])

In [ ]:
trees = pl.from_pandas(clf.get_booster().trees_to_dataframe())

In [ ]:
import polars as pl

In [ ]:
pl.Config.set_tbl_rows(100)

In [ ]:
zz = trees.filter(pl.col("Tree")==0).sort("Node", descending=True)
zz

In [ ]:
zz.write_csv("tree.csv")

In [ ]:
pwd

In [ ]:
clf.reg_lambda

In [ ]:
clf.gamma

In [ ]:
import json
tree_cfg = json.loads(clf.get_booster().save_config())

In [ ]:
tree_cfg['learner']

In [ ]:
df = pl.read_parquet("../data/Criteo/train.parquet",n_rows=10_000)
df.head()

In [ ]:
from pathlib import Path
data_dir = Path("../data/Criteo")
data_uses = ["train", "valid", "test"]
vcs = {}
for col_no in range(1,27):
    col = f"C{col_no}"
    for data_use in data_uses:
        df = pl.read_parquet(data_dir / f"{data_use}.parquet",columns=col)
        vcs[f"{col}|{data_use}"] = df.get_column(col).value_counts(sort=True)



In [ ]:
vcs.keys()

> **⚠️** `open(..., "w")` opens the file in text mode, but `pickle.dump` writes binary data — this will raise `TypeError` if run. Use `"wb"` instead.

In [ ]:
import pickle
with open(data_dir/ "categorical_enums.pkl","w") as fil:
    pickle.dump(enums, fil)

> **⚠️** `remap_parquet_files` is not defined anywhere in this repo (not in `criteo_functions.py`, not in this notebook) — this call will raise `NameError`. If the intent was to write enum-cast copies of the parquet files, something like `pl.scan_parquet(p).with_columns([pl.col(c).cast(enums[c]) for c in cats)]).sink_parquet(p.with_stem(p.stem + "_enum"))` per path would need to be written.

In [ ]:
remap_parquet_files(
    {p: p.with_stem(p.stem + "_enum") for p in paths},
    enums,
)

In [ ]:
dfs = []
for key, vc in vcs.items():
    cat, data = key.split("|")
    df = vc.select(pl.lit(cat).alias("cat"), pl.lit(data).alias("data"),pl.col(cat).alias("value"),"count")
    dfs.append(df)
df_categoricals = pl.concat(dfs)

In [ ]:
df_categoricals.head()

In [ ]:
categories = df_categoricals.drop_nulls().group_by("cat").agg(pl.col("value").unique().sort())


In [ ]:
enums = {}
for row in range(26):
    enums[categories["cat"][row]] = pl.Enum(categories["value"][row])

In [ ]:
enums["C9"]

In [ ]:
3+3

In [ ]:
dfs[0].head()

In [ ]:
vcs["C1|train"].head(20)

In [ ]:
df = get_tree_statistics(clf)

In [ ]:
df.tail()

In [ ]:
# Gamma is 8, so min is 8
df.get_column("Gain_Expected").hist(bins=[8,9,10,11,12,13,14,15,16,18,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000,5000,10000,20000])

In [ ]:
# min_child_weight=1.5,
df.get_column("H").describe(percentiles=[0.1,.2,.3,.4,.5,.6,.7,.8,.9])

In [ ]:
booster = clf.get_booster()
df = pl.from_pandas(booster.trees_to_dataframe())

lam = float(json.loads(booster.save_config())["learner"]["gradient_booster"]["tree_train_param"]["lambda"])
lam


In [ ]:
eta = float(json.loads(booster.save_config())["learner"]["gradient_booster"]["tree_train_param"]["eta"])
eta

In [ ]:
eta

In [ ]:
tree_df = get_tree_statistics(clf)

## GPU Training

### Test GPU


In [ ]:
xgb.__version__

In [ ]:
print(xgb.build_info())


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

import xgboost as xgb

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=94)

# Use "hist" for constructing the trees, with early stopping enabled.
clf = xgb.XGBClassifier(device="gpu", tree_method="hist", early_stopping_rounds=2)
# Fit the model, test sets are used for early stopping.
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)])



In [ ]:
with wandb.init(project="criteo_boost_gpu",config=config) as run:
    clf = xgb.XGBClassifier(device="gpu", tree_method="hist", early_stopping_rounds=10, min_child_weight=1.5, reg_lambda=0, gamma=8, n_estimators=200,learning_rate=.1,max_depth=60, callbacks=[WandbCallback(log_feature_importance=True)])
    clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    

In [ ]:
y_test_pred = clf.predict_proba(test_X)

6042135 test rows

In [ ]:
from sklearn.metrics import log_loss
test_log_loss = log_loss(test_y, y_test_pred)
test_log_loss


In [ ]:
tree_df = clf.get_booster().trees_to_dataframe()


In [ ]:
td.tail()

In [ ]:
td = pl.from_pandas(tree_df)

In [ ]:
td.filter(pl.col("Feature")=="Leaf").describe()

### Sweep

### Sweep

In [ ]:
sweep_config = {
    "method": "grid", # try grid or random
    "metric": {
      "name": "validation_1-logloss",
      "goal": "minimize"   
    },
    "parameters": {
        "learning_rate": {
            "values": [0.1]
        },
        "gamma": {
            "values": [0,5,10,15]
        },
        "reg_lambda": {
            "values": [0,1,10]
        },
        "min_child_weight": {
            "values": [1,2,4,8,16]
        },
        
        "max_depth":{
            "values": [50, 55, 60, 65, 70, 75, 80]
        },
    }
}

In [ ]:
sweep_id = wandb.sweep(sweep_config, project="criteo_boost_sweep_04")

In [ ]:
sweep_id = wandb.sweep(sweep_config, project="criteo_boost_sweep")

In [ ]:
sweep_id="17b17huq"

In [ ]:
wandb.teardown()

In [ ]:
wandb.agent(sweep_id, train, project="criteo_boost_sweep_04") # no count

In [ ]:
wandb.agent(sweep_id, train) # no count

In [ ]:
for sample_every in (2,):
    for cat_cutoff in (1,2,4,8,16,32):
        config = {'sample_every': sample_every, 'cat_cutoff': 25, 'cat_pseudo_count':cat_cutoff}
        train_X, train_y, valid_X, valid_y = get_data(config)
        
        with wandb.init(project="criteo_boost_sample_cutoff",config=config) as run:
            clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=10, n_estimators=150,learning_rate=.1,max_depth=60, callbacks=[WandbCallback(log_feature_importance=True)])
            clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
            tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
with wandb.init(project="criteo_boost",config=config) as run:
    clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=12, n_estimators=150,learning_rate=.1,max_depth=70, callbacks=[WandbCallback(log_feature_importance=True)])
    clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
with wandb.init(project="criteo_boost",config=config) as run:
    clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=12, n_estimators=150,learning_rate=.1,max_depth=70, callbacks=[WandbCallback(log_feature_importance=True)])
    clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    tree_df = clf.get_booster().trees_to_dataframe()
with wandb.init(project="criteo_boost",config=config) as run:
    clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=10, n_estimators=150,learning_rate=.1,max_depth=70, callbacks=[WandbCallback(log_feature_importance=True)])
    clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    tree_df = clf.get_booster().trees_to_dataframe()
with wandb.init(project="criteo_boost",config=config) as run:
    clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=10, n_estimators=150,learning_rate=.1,max_depth=80, callbacks=[WandbCallback(log_feature_importance=True)])
    clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    tree_df = clf.get_booster().trees_to_dataframe()
with wandb.init(project="criteo_boost",config=config) as run:
    clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=10, n_estimators=150,learning_rate=.1,max_depth=90, callbacks=[WandbCallback(log_feature_importance=True)])
    clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
    tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=0, gamma=10, n_estimators=150,learning_rate=.1)
clf.fit(train_X, train_y, eval_set=[(train_X, train_y),(valid_X, valid_y)],verbose=2,)
tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2, reg_lambda=10, gamma=0, n_estimators=100,learning_rate=.1)
clf.fit(train_X, train_y, eval_set=[(valid_X, valid_y)],verbose=2,)
tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
clf.learning_rate